# 06 · Reranking 对比

目标：在同一份 Chroma 向量库上，对比三种 Reranker：

| # | 方案 | 模型 | 调用方式 |
|---|------|------|----------|
| A | **BGE Reranker** | `BAAI/bge-reranker-v2-m3` | 兼容 OpenAI 的 API |
| B | **Qwen3 Reranker** | `Qwen/Qwen3-Reranker-8B` | 兼容 OpenAI 的 API |
| C | **ColBERT** | `colbert-ir/colbertv2.0` | 本地 `sentence-transformers` + MaxSim |

流程：
```
query → vector search (top-20) → Reranker → top-5 → LLM → answer
```

> 依赖：`01_data_02_chunk_ingest.ipynb` 已写入 `data/chroma`，`.env` 中已设置 `OPENAI_API_KEY` / `OPENAI_BASE_URL`。


In [2]:
# 安装依赖（首次运行）
%pip install -q sentence-transformers langchain-openai langchain-community httpx

Note: you may need to restart the kernel to use updated packages.


In [1]:
from __future__ import annotations

import os
from pathlib import Path

import httpx
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent):
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("未找到项目根目录")


def load_project_env(project_root: Path):
    for env_path in (project_root / ".env", project_root.parent / ".env"):
        if env_path.exists():
            load_dotenv(env_path, override=True)
            return env_path
    return None


PROJECT_ROOT = resolve_project_root()
ENV_FILE     = load_project_env(PROJECT_ROOT)
CHROMA_DIR   = PROJECT_ROOT / "data/chroma"
COLLECTION   = "autel_annual_report_2024"

OPENAI_API_KEY  = os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.siliconflow.cn/v1")
EMBED_MODEL     = os.getenv("EMBED_MODEL", "Qwen/Qwen3-Embedding-8B")
CHAT_MODEL      = os.getenv("CHAT_MODEL") or os.getenv("CHAT_MODEL")


# BGE / Qwen Reranker 都走同一个 OpenAI-compatible API
BGE_RERANK_MODEL  = "BAAI/bge-reranker-v2-m3"
QWEN_RERANK_MODEL = "Qwen/Qwen3-Reranker-8B"

assert CHROMA_DIR.exists(), f"找不到 Chroma 目录 {CHROMA_DIR}（先跑 01_data_02_chunk_ingest.ipynb）"
assert OPENAI_API_KEY,      f"未设置 OPENAI_API_KEY（检查 {ENV_FILE}）"

_client_kwargs = {"api_key": OPENAI_API_KEY, "base_url": OPENAI_BASE_URL}

emb = OpenAIEmbeddings(model=EMBED_MODEL, **_client_kwargs)
vs  = Chroma(collection_name=COLLECTION, embedding_function=emb,
             persist_directory=str(CHROMA_DIR))
llm = ChatOpenAI(model=CHAT_MODEL, temperature=0, **_client_kwargs)

print("ready |", COLLECTION, "| embed:", EMBED_MODEL, "| chat:", CHAT_MODEL)
print("env:", ENV_FILE)

/opt/anaconda3/envs/voc/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/opt/anaconda3/envs/voc/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/hb/4k5shxzs4c7by7lm5s0mmgrw0000gn/T/ipykernel_35829/756560120.py:49: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vs  = Chroma(collection_name=COLLECTION, embedding_function=emb,


ready | autel_annual_report_2024 | embed: Qwen/Qwen3-Embedding-8B | chat: deepseek-ai/DeepSeek-V3.2
env: /Users/mengbai/Documents/AI-training/.env


## 1. 工具函数

In [7]:
from langchain_core.documents import Document

QUERY = "道通2024年主要增长来自哪些产品线？"

RECALL_K  = 20   # 召回候选数
FINAL_K   = 5    # Reranker 保留 top-N


def recall(query: str, k: int = RECALL_K) -> list[Document]:
    """向量召回 top-k 候选"""
    return vs.similarity_search(query, k=k)


def show_docs(docs: list[Document], title: str = "", max_chars: int = 180) -> None:
    if title:
        print(f"\n{'='*60}")
        print(title)
        print('='*60)
    for i, d in enumerate(docs, 1):
        meta = {k: d.metadata.get(k) for k in ("chunk_id", "h2", "h3") if k in d.metadata}
        print(f"[{i}] {meta}")
        print(" ", d.page_content[:max_chars].replace("\n", " "))
        print()


def generate_answer(query: str, docs: list[Document]) -> str:
    context = "\n\n".join(d.page_content[:600] for d in docs)
    prompt = (
        f"根据以下上下文，简洁回答问题。\n\n上下文：\n{context}\n\n问题：{query}"
    )
    return llm.invoke(prompt).content


# ── 先跑一次召回，后面三种 Reranker 共用 ──
candidates = recall(QUERY)
print(f"召回 {len(candidates)} 条候选，query: {QUERY}")

召回 20 条候选，query: 道通2024年主要增长来自哪些产品线？


## 2. Baseline：直接用向量召回 top-5（无 Reranker）

In [8]:
baseline_docs = candidates[:FINAL_K]
show_docs(baseline_docs, title="Baseline（向量召回 top-5，无 Reranker）")

baseline_answer = generate_answer(QUERY, baseline_docs)
print("【Baseline 回答】")
print(baseline_answer)


Baseline（向量召回 top-5，无 Reranker）
[1] {'chunk_id': 234, 'h2': '1、负责任供应链'}
  公司致力于打造可持续供应链，在保障采购需求、及时履行约定的同时，积极推动供应商提升可持续发展水平，从供应商准入、采购、评价、赋能等多方面展开全流程管理，并有针对性地加入对供应商 ESG 风险的考量。

[2] {'chunk_id': 711, 'h2': '4、持续和非持续第三层次公允价值计量项目，采用的估值技术和重要参数的定性及定量信息'}
  ✓适用 ☐不适用   2024 年 12 月 31 日其他非流动金融资产（权益工具投资）中对以色列公司 Autobrains Technologies Ltd.（曾用名为 Cartica Al Ltd.）的股权投资公允价值以其持有的净资产份额计量。

[3] {'chunk_id': 108, 'h2': '2、收入和成本分析'}
  ADAS产品 | 390,456,731.32 | 155,908,300.70 | 60.07 | 26.98 | 27.9 | 减少0.29个百分点 |   其他产品 | 208,834,392.19 | 136,703,801.93 | 34.54 | 27.56 | 21.41 | 增加3.32个百分点 |   智能充电网络 | 866,700,61

[4] {'chunk_id': 760, 'h2': '6、分部信息', 'h3': '(1). 报告分部的确定依据与会计政策'}
  ✓适用 ☐不适用   公司以内部组织结构、管理要求、内部报告制度等为依据确定报告分部，并以地区分部为基础确定报告分部，分别对中国境内、北美地区、欧洲地区、其他地区等的经营业绩进行考核。

[5] {'chunk_id': 728, 'h2': '6、应收、应付关联方等未结算项目情况', 'h3': '(1). 应收项目'}
  ✓适用 ☐不适用   单位：元币种：人民币   项目名称 | 关联方 | 期末余额 | 期初余额 |   账面余额 | 坏账准备 | 账面余额 | 坏账准备 |   应收账款 | 智能航空 | 2,956,931.65 | 2,802,092.90 | 5,309,970.55 | 2,973,986.82 |   其他应收款

## 3. 方案 A：BGE Reranker（`BAAI/bge-reranker-v2-m3`）

调用硅基流动的 `/rerank` 端点（OpenAI-compatible）。

In [9]:
def api_rerank(
    query: str,
    docs: list[Document],
    model: str,
    top_n: int = FINAL_K,
) -> list[Document]:
    """
    调用 OpenAI-compatible /rerank 端点。
    硅基流动的 rerank API 格式：
      POST /rerank
      { model, query, documents: [str, ...], top_n }
    返回 { results: [{index, relevance_score}, ...] }
    """
    base = OPENAI_BASE_URL.rstrip("/")
    # SiliconFlow rerank 端点路径
    url = f"{base}/rerank"

    texts = [d.page_content[:1024] for d in docs]

    resp = httpx.post(
        url,
        headers={"Authorization": f"Bearer {OPENAI_API_KEY}",
                 "Content-Type": "application/json"},
        json={"model": model, "query": query, "documents": texts, "top_n": top_n},
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json()

    # results 按 relevance_score 降序排列，直接取
    reranked = []
    for item in data["results"]:
        idx   = item["index"]
        score = item["relevance_score"]
        doc   = docs[idx]
        doc.metadata["rerank_score"] = round(score, 4)
        reranked.append(doc)
    return reranked


bge_docs = api_rerank(QUERY, candidates, model=BGE_RERANK_MODEL)
show_docs(bge_docs, title=f"方案 A：BGE Reranker（{BGE_RERANK_MODEL}）")
for d in bge_docs:
    print(f"  chunk_id={d.metadata.get('chunk_id')}  score={d.metadata.get('rerank_score')}")

bge_answer = generate_answer(QUERY, bge_docs)
print("\n【BGE Reranker 回答】")
print(bge_answer)


方案 A：BGE Reranker（BAAI/bge-reranker-v2-m3）
[1] {'chunk_id': 214, 'h2': '4、公司环保管理制度等情况', 'h3': '(六) 有利于保护生态、防治污染、履行环境责任的相关信息'}
  ##### ✓适用 ☐不适用   公司在材料循环回收利用与可持续发展方向，非常重视循环经济的实际推进与有效落地。2024年，公司筹划并发起“Evergreen”全球ESG植树活动，携手全球六大区域的权威公益组织，深度助力逾20家战略客户巩固其全球绿色品牌形象，充分展现出行业领导者的责任担当与影响力。   从业务实践看，道通科技在美国、越南和中国的生产基地采用

[2] {'chunk_id': 312, 'h2': '(一) 股份变动情况表', 'h3': '3、股份变动对最近一年和最近一期每股收益、每股净资产等财务指标的影响（如有）'}
  ##### ✓适用 ☐不适用   报告期内，“道通转债”共有人民币32,000元已转换为公司股票，转股数量为942股，公司股本由为451,877,086股变更为451,878,028股。在归属上市公司股东的净利润不变的情况下，公司2024年度基本每股收益将相应摊薄，对公司最近一期财务状况和经营成果均不构成重大影响。

[3] {'chunk_id': 108, 'h2': '2、收入和成本分析'}
  ADAS产品 | 390,456,731.32 | 155,908,300.70 | 60.07 | 26.98 | 27.9 | 减少0.29个百分点 |   其他产品 | 208,834,392.19 | 136,703,801.93 | 34.54 | 27.56 | 21.41 | 增加3.32个百分点 |   智能充电网络 | 866,700,61

[4] {'chunk_id': 691, 'h2': '3、计入当期损益的政府补助'}
  ✓适用 ☐不适用   单位：元 币种：人民币   类型 | 本期发生额 | 上期发生额 |   与收益相关 | 69,872,885.59 | 62,779,474.07 |   与资产相关 | 842,780.59 | 976,274.67 |   合计 | 70,715,666.18 | 63,755,748.74 |  

## 4. 方案 B：Qwen3 Reranker（`Qwen/Qwen3-Reranker-8B`）

同样调用硅基流动，只换模型名。

In [10]:
qwen_docs = api_rerank(QUERY, candidates, model=QWEN_RERANK_MODEL)
show_docs(qwen_docs, title=f"方案 B：Qwen3 Reranker（{QWEN_RERANK_MODEL}）")
for d in qwen_docs:
    print(f"  chunk_id={d.metadata.get('chunk_id')}  score={d.metadata.get('rerank_score')}")

qwen_answer = generate_answer(QUERY, qwen_docs)
print("\n【Qwen3 Reranker 回答】")
print(qwen_answer)


方案 B：Qwen3 Reranker（Qwen/Qwen3-Reranker-8B）
[1] {'chunk_id': 312, 'h2': '(一) 股份变动情况表', 'h3': '3、股份变动对最近一年和最近一期每股收益、每股净资产等财务指标的影响（如有）'}
  ##### ✓适用 ☐不适用   报告期内，“道通转债”共有人民币32,000元已转换为公司股票，转股数量为942股，公司股本由为451,877,086股变更为451,878,028股。在归属上市公司股东的净利润不变的情况下，公司2024年度基本每股收益将相应摊薄，对公司最近一期财务状况和经营成果均不构成重大影响。

[2] {'chunk_id': 214, 'h2': '4、公司环保管理制度等情况', 'h3': '(六) 有利于保护生态、防治污染、履行环境责任的相关信息'}
  ##### ✓适用 ☐不适用   公司在材料循环回收利用与可持续发展方向，非常重视循环经济的实际推进与有效落地。2024年，公司筹划并发起“Evergreen”全球ESG植树活动，携手全球六大区域的权威公益组织，深度助力逾20家战略客户巩固其全球绿色品牌形象，充分展现出行业领导者的责任担当与影响力。   从业务实践看，道通科技在美国、越南和中国的生产基地采用

[3] {'chunk_id': 108, 'h2': '2、收入和成本分析'}
  ADAS产品 | 390,456,731.32 | 155,908,300.70 | 60.07 | 26.98 | 27.9 | 减少0.29个百分点 |   其他产品 | 208,834,392.19 | 136,703,801.93 | 34.54 | 27.56 | 21.41 | 增加3.32个百分点 |   智能充电网络 | 866,700,61

[4] {'chunk_id': 760, 'h2': '6、分部信息', 'h3': '(1). 报告分部的确定依据与会计政策'}
  ✓适用 ☐不适用   公司以内部组织结构、管理要求、内部报告制度等为依据确定报告分部，并以地区分部为基础确定报告分部，分别对中国境内、北美地区、欧洲地区、其他地区等的经营业绩进行考核。

[5] {'chunk_id': 35, 'h2': '目录', 'h3': '十二、因国家秘

## 5. 方案 C：ColBERT（`colbert-ir/colbertv2.0`，本地 sentence-transformers）

ColBERT 使用 **Late Interaction**：query 和 document **分别**编码为 token 级别向量，  
通过 **MaxSim**（每个 query token 找最相似的 doc token，再求和）计算相关性得分。

```
score(q, d) = Σ_{i∈q} max_{j∈d} ( q_i · d_j )
```

与 Cross-Encoder 的区别：
- Cross-Encoder：query + doc 拼接，一次前向，精度高但无法预计算 doc 侧
- ColBERT：doc 侧可**离线预计算并缓存** token embeddings，查询时只需编码 query，延迟更低

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

# jina-colbert-v2：多语言 ColBERT，中文效果远好于英文版 colbertv2.0
colbert_model = SentenceTransformer("colbert-ir/colbertv2.0", trust_remote_code=True)
print(f"ColBERT 加载完成 | model: colbert-ir/colbertv2.0 | device: {colbert_model.device}")

No sentence-transformers model found with name colbert-ir/colbertv2.0. Creating a new one with mean pooling.


In [ ]:
def maxsim(q_emb: torch.Tensor, d_emb: torch.Tensor) -> float:
    """
    ColBERT Late Interaction MaxSim：
      q_emb: (q_len, dim)  — query token embeddings
      d_emb: (d_len, dim)  — doc   token embeddings
    score = Σ_{i∈q} max_{j∈d} cosine_sim(q_i, d_j)
    """
    # (q_len, d_len) 相似度矩阵
    sim_matrix = torch.matmul(q_emb, d_emb.T)
    # 每个 query token 取最相似 doc token，再对所有 query token 求和
    return sim_matrix.max(dim=1).values.sum().item()


def colbert_rerank(
    query: str,
    docs: list[Document],
    top_n: int = FINAL_K,
) -> list[Document]:
    texts = [d.page_content[:512] for d in docs]   # ColBERT 默认 max 512 tokens

    # 分别编码 query 和每条 doc，获取 token 级别向量
    # output_value="token_embeddings" 返回 list[Tensor(seq_len, dim)]
    q_token_embs = colbert_model.encode(
        [query],
        output_value="token_embeddings",
        convert_to_tensor=True,
        normalize_embeddings=True,
    )[0]  # (q_len, dim)

    d_token_embs = colbert_model.encode(
        texts,
        output_value="token_embeddings",
        convert_to_tensor=True,
        normalize_embeddings=True,
        batch_size=8,
        show_progress_bar=False,
    )  # list of Tensor(d_len, dim)

    # 计算每条 doc 的 MaxSim 得分
    scored = []
    for doc, d_emb in zip(docs, d_token_embs):
        score = maxsim(q_token_embs, d_emb)
        scored.append((score, doc))

    # 按得分降序排列，取 top_n
    scored.sort(key=lambda x: x[0], reverse=True)
    reranked = []
    for rank, (score, doc) in enumerate(scored[:top_n], 1):
        doc.metadata["rerank_score"] = round(score, 4)
        doc.metadata["rerank_rank"]  = rank
        reranked.append(doc)
    return reranked


colbert_docs = colbert_rerank(QUERY, candidates)
show_docs(colbert_docs, title="方案 C：ColBERT（colbert-ir/colbertv2.0，sentence-transformers + MaxSim）")
for d in colbert_docs:
    print(f"  chunk_id={d.metadata.get('chunk_id')}  score={d.metadata.get('rerank_score')}  rank={d.metadata.get('rerank_rank')}")

colbert_answer = generate_answer(QUERY, colbert_docs)
print("\n【ColBERT Reranker 回答】")
print(colbert_answer)

NameError: name 'colbert_model' is not defined

## 6. 三方案对比汇总

In [ ]:
import pandas as pd

def summarize(label: str, docs: list[Document]) -> list[dict]:
    rows = []
    for rank, d in enumerate(docs, 1):
        rows.append({
            "方案": label,
            "rank": rank,
            "chunk_id": d.metadata.get("chunk_id", ""),
            "score": d.metadata.get("rerank_score", "-"),
            "h2": d.metadata.get("h2", "")[:30],
            "text_preview": d.page_content[:80].replace("\n", " "),
        })
    return rows


rows = (
    summarize("Baseline（无Rerank）", baseline_docs)
    + summarize("BGE v2-m3", bge_docs)
    + summarize("Qwen3-Reranker-8B", qwen_docs)
    + summarize("ColBERT v2.0", colbert_docs)
)

df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 25)
df

In [ ]:
# chunk_id 集合对比：三种 Reranker 各自留下了哪些 chunk？
baseline_ids = {d.metadata.get("chunk_id") for d in baseline_docs}
bge_ids      = {d.metadata.get("chunk_id") for d in bge_docs}
qwen_ids     = {d.metadata.get("chunk_id") for d in qwen_docs}
colbert_ids  = {d.metadata.get("chunk_id") for d in colbert_docs}

print("Baseline  chunk_ids :", sorted(baseline_ids))
print("BGE       chunk_ids :", sorted(bge_ids))
print("Qwen3     chunk_ids :", sorted(qwen_ids))
print("ColBERT   chunk_ids :", sorted(colbert_ids))
print()
print("BGE ∩ Qwen3 ∩ ColBERT（三者都认为相关）:",
      sorted(bge_ids & qwen_ids & colbert_ids))
print("BGE △ ColBERT（BGE 选了但 ColBERT 没选，或反之）:",
      sorted(bge_ids ^ colbert_ids))

## 7. 机制小结

| | BGE Reranker | Qwen3 Reranker | ColBERT |
|---|---|---|---|
| **架构** | Cross-Encoder | Cross-Encoder | Late Interaction |
| **编码方式** | query + doc 拼接后一次前向 | 同左 | query / doc **分别**编码，token 级 MaxSim |
| **调用方式** | API（硅基流动） | API（硅基流动） | 本地 `sentence-transformers` |
| **doc 侧可预计算** | ❌ | ❌ | ✅（可离线缓存 token embeddings） |
| **速度** | 取决于网络 + 并发 | 同左 | 本地推理，高 QPS 场景更可控 |
| **中文效果** | 强（m3 多语言） | 强（Qwen 中文预训练） | 偏英文，中文需 fine-tune |
| **适用场景** | 中英混合企业文档 | 纯中文知识库 | 英文为主，或需要离线高速 rerank |

**工程建议**：
- 中文场景优先评估 `Qwen3-Reranker-8B`，再对比 `BGE-reranker-v2-m3`，选 MRR@5 更高的；  
- ColBERT 最大的工程优势是 **doc token embeddings 可离线预计算并缓存**，查询时只需编码 query → 大幅降低在线延迟，适合高 QPS；  
- 本 notebook 的 ColBERT 是"纯 rerank"模式（不缓存），生产中应先对全库 doc 做离线编码，存 `Tensor` 到磁盘/向量库，查询时直接 load；  
- 生产建议：Hybrid（BM25 + Dense）召回 top-50，再用 Cross-Encoder 精排 top-5，延迟可控在 300ms 以内。
